#### # 1. Multi-Table Ingestion & Silver Staging Setup

In [0]:

import pyspark.sql.functions as F

# Ingest tables with the precise aliases recognized by your Spark catalog
crm_cust = spark.table("workspace.silver.crm_customers").alias("ci")
erp_cust = spark.table("workspace.silver.erp_customer").alias("ca")
erp_loc  = spark.table("workspace.silver.erp_location").alias("lo")



#### # 2. Relational Joins, Windowing & Master Records Logic

In [0]:
from pyspark.sql.window import Window

# 1. Execute Left Joins utilizing your verified catalog column structures
joined_df = (
    crm_cust
    .join(erp_cust, F.col("ci.customer_number") == F.col("ca.customer_id"), "left")
    .join(erp_loc,  F.col("ci.customer_number") == F.col("lo.customer_id"), "left")
)

# 2. Process analytical attributes, master business routing, and surrogate keys
transformed_df = (
    joined_df
    # Build your sequential surrogate key tracking over the primary customer id
    .withColumn("customer_key", F.row_number().over(Window.orderBy("ci.customer_id")))
    
    # Master logic: CRM is the master source for gender. Fall back to ERP if CRM is 'n/a'
    .withColumn(
        "gender_cleansed",
        F.when(F.lower(F.col("ci.gender")) != "n/a", F.col("ci.gender"))
        .otherwise(F.coalesce(F.col("ca.gender"), F.lit("N/A")))
    )
)

#### #3. Dimensional Schema Ordering, Casting, and Gold Table Storage

In [0]:
# Single-Pass Operation: Extracts specific alias attributes, enforces type-casting, and outputs final names
final_df = transformed_df.select(
    F.col("customer_key").cast("integer").alias("customer_key"),
    F.col("ci.customer_id").cast("string").alias("customer_id"),
    F.col("ci.customer_number").cast("string").alias("customer_number"),
    F.col("ci.first_name").cast("string").alias("first_name"),
    F.col("ci.last_name").cast("string").alias("last_name"),
    F.col("lo.country").cast("string").alias("country"), # Updated alias target to lo
    F.col("ci.marital_status").cast("string").alias("marital_status"),
    F.col("gender_cleansed").cast("string").alias("gender"),
    F.col("ca.birth_date").cast("date").alias("birthdate"),
    F.col("ci.created_date").cast("timestamp").alias("create_date")
)

# Commit the finalized schema directly as a production-grade Gold Delta table
final_df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_customers")

# Display a clean data preview to inspect column alignment and verify rows
final_df.limit(10).display()